# Download, preparação e inspeção de dados (NIDDK - Diabetes Dataset)

<mark>Este notebook é didático e não deve ser utilizado para realizar diagnósticos</mark>

## Sobre o conjunto de dados

Este conjunto de dados tem como origem o [National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK)](https://www.niddk.nih.gov/) para estudos relacionados ao diabetes. Não assumimos que os registros sejam representativos de uma população clínica.

Os dados estão disponíveis no repositório [MicrosoftLearning/AI-900-AIFundamentals](https://github.com/MicrosoftLearning/AI-900-AIFundamentals/blob/main/data/ml/diabetes.csv), que é registrado como o canal de download.

## Bibliotecas e caminhos

As pastas são criadas automaticamente. O Parquet e os arquivos de origem ficam em `data/`, ignorada pelo Git.

### Bibliotecas

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import io
import json
import zipfile

import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")


def find_project_root():
    """Procura o diretório raiz do projeto."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "ml" / "pyproject.toml").is_file() and (
            candidate / "README.md"
        ).is_file():
            return candidate
    raise FileNotFoundError("Abra o notebook dentro deste repositório.")

### Variáveis

In [ ]:
PROJECT_ROOT = find_project_root()

DATASET_URL = "https://raw.githubusercontent.com/MicrosoftLearning/AI-900-AIFundamentals/main/data/ml/diabetes.csv"
SOURCE_PAGE = "https://www.niddk.nih.gov/"
DISTRIBUTION_PAGE = "https://github.com/MicrosoftLearning/AI-900-AIFundamentals/blob/main/data/ml/diabetes.csv"
DATA_DIR = PROJECT_ROOT / "data" / "niddk-diabetes"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = DATA_DIR / "diabetes.csv"
PARQUET_PATH = DATA_DIR / "diabetes.parquet"
METADATA_PATH = DATA_DIR / "source.json"

COLUMN_TRANSLATIONS = {
    "PatientID": "id_paciente",
    "Pregnancies": "gestacoes",
    "PlasmaGlucose": "glicose_plasmatica",
    "DiastolicBloodPressure": "pressao_arterial_diastolica",
    "TricepsThickness": "espessura_triceps",
    "SerumInsulin": "insulina_serica",
    "BMI": "imc",
    "DiabetesPedigree": "indice_hereditariedade_diabetes",
    "Age": "idade",
    "Diabetic": "diabetes",
}

RAW_TARGET = "Diabetic"
TARGET = COLUMN_TRANSLATIONS[RAW_TARGET]
EXPECTED_COLUMNS = [
    "PatientID",
    "Pregnancies",
    "PlasmaGlucose",
    "DiastolicBloodPressure",
    "TricepsThickness",
    "SerumInsulin",
    "BMI",
    "DiabetesPedigree",
    "Age",
    "Diabetic",
]
FORCE_DOWNLOAD = False

print(f"Pasta de dados: {DATA_DIR.relative_to(PROJECT_ROOT).as_posix()}")

## Download e leitura

O arquivo existente é reutilizado. Para baixar novamente, altere `FORCE_DOWNLOAD`. O conteúdo recebido é validado antes de substituir o CSV local; uma falha de rede pode ser corrigida executando a célula novamente.

In [ ]:
if FORCE_DOWNLOAD or not CSV_PATH.is_file():

    response = requests.get(DATASET_URL, timeout=(15, 60))
    response.raise_for_status()

    csv_bytes = response.content
    candidate = pd.read_csv(io.BytesIO(csv_bytes))

    if candidate.empty or list(candidate.columns) != EXPECTED_COLUMNS:
        raise ValueError(
            "O arquivo recebido está vazio ou seu esquema mudou. Confira a fonte."
        )

    temporary_path = CSV_PATH.with_suffix(".csv.part")
    temporary_path.write_bytes(csv_bytes)
    temporary_path.replace(CSV_PATH)

    metadata = {
        "source_page": SOURCE_PAGE,
        "distribution_page": DISTRIBUTION_PAGE,
        "download_url": DATASET_URL,
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
        "csv_sha256": hashlib.sha256(csv_bytes).hexdigest(),
    }

    METADATA_PATH.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("Download concluído.")
else:
    print("Usando o CSV já disponível.")

data = pd.read_csv(CSV_PATH)

if data.empty or list(data.columns) != EXPECTED_COLUMNS:
    raise ValueError("CSV local vazio ou com colunas inesperadas. Confira o arquivo.")

if RAW_TARGET not in data:
    raise ValueError("Coluna alvo ausente.")

expected_classes = {0, 1}

if data[RAW_TARGET].isna().any() or set(data[RAW_TARGET].unique()) != expected_classes:
    raise ValueError("A coluna alvo contém valores ausentes ou classes inesperadas.")

print("Visualização do conjunto de dados original:")
display(data.head())

## Preparação e salvamento em Parquet

Abaixo, o conjunto de dados original é transformado: as colunas são traduzidas para PT-BR e normalizadas em `lowercase` com `_` como separador. A variável alvo já está codificada como `0`/`1` e é armazenada como `int8`. A prévia permite comparar o resultado com o CSV original exibido na seção anterior. Não removemos duplicatas, imputamos ausências ou normalizamos atributos. Essas decisões ficam para os experimentos posteriores.

In [ ]:
data = data.rename(columns=COLUMN_TRANSLATIONS)
data[TARGET] = data[TARGET].astype("int8")

print("Visualização do conjunto de dados preparado:")
display(data.head())

temporary_parquet = PARQUET_PATH.with_suffix(".parquet.part")
data.to_parquet(temporary_parquet, engine="pyarrow", index=False)
restored = pd.read_parquet(temporary_parquet, engine="pyarrow")
pd.testing.assert_frame_equal(data, restored)
temporary_parquet.replace(PARQUET_PATH)

print(f"Parquet salvo e conferido: {PARQUET_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"{len(data):,} registros; {len(data.columns)} colunas.")

## Estrutura e qualidade dos dados

Observe tipos, valores ausentes e duplicatas. Registros iguais não são automaticamente erros; avalie seu significado antes de removê-los.

In [ ]:
data.info()

In [ ]:
quality = pd.DataFrame({
    "tipo": data.dtypes.astype(str),
    "ausentes": data.isna().sum(),
    "ausentes_percentual": data.isna().mean().mul(100).round(2),
    "valores_distintos": data.nunique(dropna=False),
})

display(quality)

In [ ]:
print(f"Linhas inteiramente duplicadas: {data.duplicated().sum()}")

In [ ]:
print(f"id_paciente duplicados: {data['id_paciente'].duplicated().sum()}")

In [ ]:
print(f"Duplicatas sem considerar id_paciente: {data.drop(columns='id_paciente').duplicated().sum()}")

In [ ]:
display(data.describe(include="all").T)

## Balanceamento das classes

As proporções são calculadas diretamente do arquivo. O balanceamento influencia a escolha de métricas e a futura separação dos dados.

In [ ]:
class_labels = {0: "Negativo (0)", 1: "Positivo (1)"}

counts = data[TARGET].value_counts().sort_index()

balance = pd.DataFrame(
    {"quantidade": counts, "percentual": counts.div(len(data)).mul(100).round(2)}
)
balance.index = [class_labels[int(value)] for value in balance.index]

display(balance)

In [ ]:
ax = sns.barplot(x=balance.index, y=balance["quantidade"], color="#4c72b0")
ax.set(title="Distribuição das classes", xlabel="Classe", ylabel="Número de registros")

plt.tight_layout()
plt.show()

## Distribuição das variáveis

Os gráficos são descritivos: diferenças entre grupos e correlações não demonstram causalidade.

In [ ]:
numeric_columns = [
    name
    for name in data.select_dtypes(include="number").columns
    if name not in {TARGET, "id_paciente"}
]

if numeric_columns:
    data[numeric_columns].hist(
        bins=25,
        figsize=(12, max(3, 3 * ((len(numeric_columns) + 2) // 3))),
        layout=((len(numeric_columns) + 2) // 3, 3),
    )
    plt.suptitle("Distribuição das variáveis numéricas")
    plt.tight_layout()
    plt.show()

if "idade" in data:
    sns.boxplot(data=data, x=TARGET, y="idade")
    plt.title("Idade por classe")
    plt.tight_layout()
    plt.show()

### Zeros e correlações

Verificamos a frequência de zeros nas medidas abaixo, sem convertê-los automaticamente em valores ausentes. A interpretação exige conferir o significado de cada campo na fonte. `id_paciente` é excluído da matriz de correlação.

In [ ]:
measurement_columns = [
    "glicose_plasmatica",
    "pressao_arterial_diastolica",
    "espessura_triceps",
    "insulina_serica",
    "imc",
]

zeros = data[measurement_columns].eq(0).sum()

display(
    pd.DataFrame({"zeros": zeros, "percentual": zeros.div(len(data)).mul(100).round(2)})
)

In [ ]:
correlation_columns = numeric_columns + [TARGET]

sns.heatmap(
    data[correlation_columns].corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f"
)

plt.title("Correlação de Pearson entre variáveis numéricas")
plt.tight_layout()
plt.show()